# Uncovered Interest Rate Parity (UIP): USA vs. Eurozone
## Empirische Überprüfung anhand von Quartalsdaten 2022–2026

---

**Kurs:** Fundamentals of Exchange Rate Theory and Exchange Rate Markets  
**Niveau:** Bachelor Economics

---

## Wie du dieses Notebook benutzt

Du musst **keinen Code selbst schreiben**. Deine Aufgabe ist es, jeden Schritt zu **verstehen und zu interpretieren**.

**Zelle ausführen:** Klicke auf die Zelle, dann drücke **Shift + Enter**  
**Wichtig:** Führe die Zellen immer **von oben nach unten** aus!

---

## 1. Theoretischer Hintergrund

### Was sagt die Uncovered Interest Rate Parity (UIP)?

Stell dir vor, du hast 1.000 Euro und kannst zwischen zwei Anlagen wählen:

**Option A – Anlage in der Eurozone:**  
Du legst 1.000 Euro zum Quartalszinssatz an:
91.000 \cdot \left(1 + rac{i_{EU}}{400}ight)9

**Option B – Anlage in den USA:**  
Du tauschst zuerst in Dollar (Kurs $: USD pro EUR), legst zum US-Zinssatz an, tauschst am Ende zurück:
91.000 \cdot S_t \cdot \left(1 + rac{i_{US}}{400}ight) \cdot rac{1}{S_{t+1}}9

Die **UIP-Bedingung** besagt: Im Gleichgewicht sind beide Optionen gleich attraktiv:

9oxed{S_{t+1}^{	ext{UIP}} = S_t \cdot rac{1 + i_{US}/400}{1 + i_{EU}/400}}9

**Intuition:** Wenn US-Zinsen höher sind als EU-Zinsen, muss der Dollar laut UIP **abwerten**, damit der Zinsvorteil ausgeglichen wird.

### Datenbasis

| Variable | FRED-Kürzel | Beschreibung |
|---|---|---|
| {US}$ |  | 3-Month Treasury Bill Rate, USA |
| {EU}$ |  | 3-Month Interbank Rate, Deutschland |
| $ |  | USD pro EUR, Spot-Wechselkurs |

---

## 2. Setup: Pakete laden

Führe diese Zelle zuerst aus. Sie lädt alle benötigten R-Pakete.

In [ ]:
required_packages <- c("dplyr", "ggplot2", "lubridate", "tidyr", "scales", "knitr")

for (pkg in required_packages) {
  if (!require(pkg, character.only = TRUE, quietly = TRUE)) {
    install.packages(pkg, repos = "https://cran.rstudio.com/", quiet = TRUE)
    library(pkg, character.only = TRUE, quietly = TRUE)
  }
}

options(repr.plot.width = 10, repr.plot.height = 5)
cat("Alle Pakete geladen!
")

---

## 3. Daten laden

Die Daten der drei FRED-Zeitreihen liegen direkt im Notebook als CSV-Dateien vor – **kein Internet, kein Account, kein API-Key nötig**.

| Variable | Datei | Beschreibung |
|---|---|---|
| $i_{US}$ | `DTB3.csv` | 3-Month Treasury Bill Rate, USA |
| $i_{EU}$ | `IR3TIB01DEM156N.csv` | 3-Month Interbank Rate, Deutschland |
| $S_t$ | `DEXUSEU.csv` | USD pro EUR, Spot-Wechselkurs |

In [ ]:
# Daten aus den lokalen CSV-Dateien laden
# (Dateien liegen direkt im Repository - kein Internet nötig)

us_raw <- read.csv("DTB3.csv")
eu_raw <- read.csv("IR3TIB01DEM156N.csv")
fx_raw <- read.csv("DEXUSEU.csv")

colnames(us_raw) <- c("date", "value")
colnames(eu_raw) <- c("date", "value")
colnames(fx_raw) <- c("date", "value")

clean <- function(df) {
  df |>
    dplyr::mutate(date  = as.Date(date),
                  value = suppressWarnings(as.numeric(value))) |>
    dplyr::filter(!is.na(value))
}

us_raw <- clean(us_raw)
eu_raw <- clean(eu_raw)
fx_raw <- clean(fx_raw)

cat("Daten erfolgreich geladen!\n")
cat("  US-Zinssatz:  ", nrow(us_raw), "Beobachtungen\n")
cat("  EU-Zinssatz:  ", nrow(eu_raw), "Beobachtungen\n")
cat("  Wechselkurs:  ", nrow(fx_raw), "Beobachtungen\n")

---

## 4. Datenaufbereitung: Quartalswerte

Wir nehmen jeweils den **ersten Monat** jedes Quartals (Jan, Apr, Jul, Okt) als Startzinssatz
und vergleichen ihn mit dem Wechselkurs **am Ende desselben Quartals**.

In [ ]:
# Analysezeitraum
start_date <- as.Date("2022-01-01")
end_date   <- as.Date("2025-12-31")

# Hilfsfunktion: Monatsmittelwert, dann Quartalsbeginn filtern
to_quarterly <- function(df, varname) {
  df |>
    filter(date >= start_date, date <= end_date) |>
    mutate(month_start = floor_date(date, "month")) |>
    group_by(month_start) |>
    summarise(value = mean(value, na.rm = TRUE), .groups = "drop") |>
    mutate(qstart = floor_date(month_start, "quarter")) |>
    filter(month_start == qstart) |>
    select(qstart, value) |>
    rename(!!varname := value)
}

us_q <- to_quarterly(us_raw, "i_US")
eu_q <- to_quarterly(eu_raw, "i_EU")
fx_q <- to_quarterly(fx_raw, "S_start")

# Wechselkurs Quartalsende
fx_end_raw <- fx_raw |>
  filter(date >= start_date, date <= end_date + months(3)) |>
  mutate(month_start = floor_date(date, "month")) |>
  group_by(month_start) |>
  summarise(value = mean(value, na.rm = TRUE), .groups = "drop") |>
  mutate(monthnum = as.integer(format(month_start, "%m")),
         qstart   = floor_date(month_start, "quarter")) |>
  filter(monthnum %in% c(1, 4, 7, 10)) |>
  mutate(qstart = qstart - months(3)) |>
  filter(qstart >= start_date) |>
  select(qstart, value) |>
  rename(S_end = value)

# Alle Variablen zusammenfuehren
df <- us_q |>
  inner_join(eu_q,       by = "qstart") |>
  inner_join(fx_q,       by = "qstart") |>
  inner_join(fx_end_raw, by = "qstart") |>
  filter(!is.na(i_US), !is.na(i_EU), !is.na(S_start), !is.na(S_end))

cat("Quartalsdatensatz erstellt:", nrow(df), "Quartale\n")
cat("Zeitraum:", format(min(df$qstart)), "bis", format(max(df$qstart)), "\n")

---

## 5. Rohdaten: Ein erster Blick

In [ ]:
df |>
  mutate(Quartal = paste0(year(qstart), " Q", quarter(qstart))) |>
  select(Quartal,
         `i_US (% p.a.)` = i_US,
         `i_EU (% p.a.)` = i_EU,
         `S Anfang (USD/EUR)` = S_start,
         `S Ende (USD/EUR)`   = S_end) |>
  knitr::kable(digits = 3, align = "lcccc")

---

## 6. Zinsentwicklung im Zeitverlauf

In [ ]:
library(ggplot2)
library(tidyr)

df |>
  select(qstart, i_US, i_EU) |>
  pivot_longer(c(i_US, i_EU), names_to = "Region", values_to = "Zinssatz") |>
  mutate(Region = dplyr::recode(Region,
    i_US = "USA (3M Treasury Bill)",
    i_EU = "Eurozone (3M Interbank DE)"
  )) |>
ggplot(aes(x = qstart, y = Zinssatz, color = Region, linetype = Region)) +
  geom_line(linewidth = 1.3) +
  geom_point(size = 3) +
  scale_color_manual(values = c(
    "USA (3M Treasury Bill)"     = "#1f77b4",
    "Eurozone (3M Interbank DE)" = "#d62728"
  )) +
  scale_x_date(date_breaks = "6 months", date_labels = "%b %Y") +
  labs(title = "3-Monats-Zinssätze: USA vs. Eurozone",
       subtitle = "Quartalsbeginnswerte | Quelle: FRED",
       x = NULL, y = "Zinssatz (% p.a.)", color = NULL, linetype = NULL) +
  theme_minimal(base_size = 13) +
  theme(legend.position = "bottom", axis.text.x = element_text(angle = 30, hjust = 1))

> ### Aufgabe 1: Zinsentwicklung interpretieren
>
> **a)** In welchem Zeitraum waren die US-Zinsen deutlich höher als die EU-Zinsen?
>
> **b)** Was sagt die UIP für diesen Zeitraum: Sollte der Dollar ab- oder aufwerten?
>
> **c)** Erkennst du eine Annäherung der Zinssätze? Wann findet diese statt?

**Deine Antworten zu Aufgabe 1:**

a) ...

b) ...

c) ...

---

## 7. UIP-Berechnung

9\boxed{S_{t+1}^{\text{UIP}} = S_t \cdot \frac{1 + i_{US}/400}{1 + i_{EU}/400}}9

In [ ]:
df <- df |>
  mutate(
    zins_diff      = i_US - i_EU,
    S_UIP          = S_start * (1 + i_US / 400) / (1 + i_EU / 400),
    delta_s_actual = (S_end  / S_start - 1) * 100,
    delta_s_uip    = (S_UIP  / S_start - 1) * 100,
    abweichung     = delta_s_actual - delta_s_uip
  )

df |>
  mutate(Quartal = paste0(year(qstart), " Q", quarter(qstart))) |>
  select(Quartal,
         `Zinsdiff.(PP)` = zins_diff,
         `S Anfang`      = S_start,
         `S UIP`         = S_UIP,
         `S tatsächl.`   = S_end,
         `Abw.(%)`       = abweichung) |>
  knitr::kable(digits = 4, align = "lccccc")

---

## 8. UIP-Prognose vs. tatsächlicher Wechselkurs

In [ ]:
df |>
  select(qstart, S_start, S_UIP, S_end) |>
  pivot_longer(c(S_start, S_UIP, S_end), names_to = "Typ", values_to = "Kurs") |>
  mutate(Typ = dplyr::recode(Typ,
    S_start = "Spot-Rate (Quartalsbeginn)",
    S_UIP   = "UIP-Prognose (Quartalsende)",
    S_end   = "Tatsächl. Rate (Quartalsende)"
  )) |>
ggplot(aes(x = qstart, y = Kurs, color = Typ, linetype = Typ, shape = Typ)) +
  geom_line(linewidth = 1.1) +
  geom_point(size = 3) +
  scale_color_manual(values = c(
    "Spot-Rate (Quartalsbeginn)"    = "#7f7f7f",
    "UIP-Prognose (Quartalsende)"   = "#2ca02c",
    "Tatsächl. Rate (Quartalsende)" = "#d62728"
  )) +
  scale_x_date(date_breaks = "6 months", date_labels = "%b %Y") +
  labs(title = "UIP-Prognose vs. tatsächlicher USD/EUR-Wechselkurs",
       subtitle = "Quartalswerte | Quelle: FRED",
       x = NULL, y = "USD pro EUR", color = NULL, linetype = NULL, shape = NULL) +
  theme_minimal(base_size = 13) +
  theme(legend.position = "bottom", axis.text.x = element_text(angle = 30, hjust = 1))

> ### Aufgabe 2: UIP-Prognose vs. Realität
>
> **a)** In welchem Zeitraum liegt der tatsächliche Kurs deutlich über der UIP-Prognose?
>
> **b)** Gibt es Phasen, in denen die UIP-Prognose relativ gut abschneidet?
>
> **c)** Was sagen dir die Abstände zwischen grüner und roter Linie?

**Deine Antworten zu Aufgabe 2:**

a) ...

b) ...

c) ...

---

## 9. Abweichungen von der UIP

In [ ]:
ggplot(df, aes(x = qstart, y = abweichung, fill = abweichung > 0)) +
  geom_col(width = 70, show.legend = FALSE) +
  geom_hline(yintercept = 0, linewidth = 0.8) +
  scale_fill_manual(values = c("TRUE" = "#1f77b4", "FALSE" = "#d62728")) +
  scale_x_date(date_breaks = "6 months", date_labels = "%b %Y") +
  labs(title = "Abweichung der tatsächlichen Kursänderung von der UIP-Prognose",
       subtitle = "Blau: USD stärker abgewertet als UIP prognostiziert | Rot: USD weniger abgewertet",
       x = NULL, y = "Abweichung (Prozentpunkte)") +
  theme_minimal(base_size = 13) +
  theme(axis.text.x = element_text(angle = 30, hjust = 1))

---

## 10. Statistischer UIP-Test: Regression

Der klassische empirische Test der UIP:

9\Delta s_{t+1} = \alpha + \beta \cdot (i_{US,t} - i_{EU,t}) + \varepsilon_{t+1}9

**UIP erwartet:** $\alpha = 0$ und $\beta = 0.25$ (quartalsweise)

In [ ]:
reg <- lm(delta_s_actual ~ zins_diff, data = df)
summary(reg)

In [ ]:
beta_hat  <- coef(reg)[2]

ggplot(df, aes(x = zins_diff, y = delta_s_actual)) +
  geom_point(size = 3.5, color = "#1f77b4", alpha = 0.8) +
  geom_text(aes(label = paste0(year(qstart), " Q", quarter(qstart))),
            vjust = -0.8, size = 3) +
  geom_smooth(method = "lm", se = TRUE, color = "#d62728", fill = "#f7b6b6") +
  geom_abline(intercept = 0, slope = 0.25,
              linetype = "dashed", color = "#2ca02c", linewidth = 1.2) +
  annotate("text", x = Inf, y = Inf,
           label = paste0("OLS: beta = ", round(beta_hat, 3)),
           hjust = 1.1, vjust = 1.5, size = 4.5, color = "#d62728") +
  annotate("text", x = Inf, y = -Inf,
           label = "UIP: beta = 0.25",
           hjust = 1.1, vjust = -0.5, size = 4.5, color = "#2ca02c") +
  labs(title = "UIP-Test: Zinsdifferenzial vs. tatsächliche Wechselkursänderung",
       subtitle = "Rote Linie: OLS | Grün gestrichelt: UIP-Prognose (beta = 0.25)",
       x = "Zinsdifferenzial i_US - i_EU (Prozentpunkte, Jahresbasis)",
       y = "Tatsächliche Wechselkursänderung (%, USD/EUR)") +
  theme_minimal(base_size = 13)

> ### Aufgabe 3: Regressionsergebnisse interpretieren
>
> **a)** Welchen Wert hat beta? Ist er signifikant von Null verschieden?
>
> **b)** Was bedeutet es, wenn beta negativ ist? Was ist das Forward Premium Puzzle?
>
> **c)** Was sagt der R²-Wert über die Erklärungskraft der UIP aus?
>
> **d)** Nenne zwei mögliche Erklärungen warum die UIP empirisch scheitern könnte.

**Deine Antworten zu Aufgabe 3:**

a) ...

b) ...

c) ...

d) ...

---

## 11. Zusammenfassung

| Konzept | Kernaussage |
|---|---|
| **UIP-Formel** | {t+1}^{UIP} = S_t \cdot \frac{1+i_{US}/400}{1+i_{EU}/400}$ |
| **Zinsdifferenzial** | USA hatte 2022–2023 deutlich höhere Zinsen als die Eurozone |
| **Forward Premium Puzzle** | Hochzinswährungen werten empirisch häufig auf, nicht ab |

---

### Aufgabe 4 (Abschlussreflexion)

Formuliere in **3–5 Sätzen**: Was bedeutet das Ergebnis für einen Investor, der auf Basis der UIP spekulieren will?

**Deine Antwort zu Aufgabe 4:**

...

---

*Datenquelle: Federal Reserve Bank of St. Louis (FRED) — fred.stlouisfed.org*  
*Reihen: DTB3, IR3TIB01DEM156N, DEXUSEU*